# Module 09 — Notebook 3: Inter-Rater Agreement

## Learning Objectives

By the end of this notebook, you will be able to:

- Explain why human annotator disagreement threatens eval validity
- Compute percent agreement between two raters
- Compute Cohen's kappa from scratch using Python dicts and lists
- Interpret a kappa value using the standard rule-of-thumb thresholds

**Estimated time:** ~20 minutes

## Why This Matters for AI Research Engineering

Human evaluation is the gold standard for judging model outputs — especially for safety, honesty, and helpfulness, where there's no single right answer. But humans disagree. If your annotators can't agree on what counts as "helpful," your eval is measuring annotator noise, not model quality.

Inter-rater agreement analysis is how you audit your rubric before you scale up annotation. Low kappa tells you to fix your rubric; high kappa tells you your labels are trustworthy. This is a standard step in every serious AI evaluation project.

In [ ]:
import sys
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_approx, check_contains, check_length
print("Setup complete.")

## 1. Why Raters Disagree

When two people rate the same model output, they might disagree because:

- **Ambiguous rubric:** the instructions don't specify what to do in edge cases
- **Different interpretation of the scale:** one rater's "3" is another's "4"
- **Personal values:** what counts as "harmful" or "helpful" varies by perspective
- **Attention and fatigue:** raters make random errors, especially late in a long session

The first two are fixable with better rubric design. The last two are inherent to human annotation.

Before you trust your labels, you need to measure how much agreement you actually have.

In [ ]:
# Two annotators rating 20 model outputs as "pass" or "fail"
# (Would this response be acceptable in a production AI assistant?)

rater_1 = ["pass", "pass", "fail", "pass", "pass",
           "fail", "pass", "fail", "pass", "pass",
           "pass", "fail", "pass", "pass", "fail",
           "pass", "fail", "pass", "pass", "fail"]

rater_2 = ["pass", "pass", "fail", "fail", "pass",
           "fail", "pass", "pass", "pass", "pass",
           "pass", "fail", "fail", "pass", "fail",
           "pass", "fail", "pass", "pass", "pass"]

print(f"Number of items rated: {len(rater_1)}")
print(f"Rater 1 — pass: {rater_1.count('pass')}, fail: {rater_1.count('fail')}")
print(f"Rater 2 — pass: {rater_2.count('pass')}, fail: {rater_2.count('fail')}")

## 2. Percent Agreement

The simplest measure of inter-rater agreement: what fraction of items did both raters label the same way?

```
percent_agreement = number_of_matching_labels / total_items
```

**The problem with percent agreement:** it doesn't account for chance. If raters are both guessing randomly on a 50/50 task, they'll agree ~50% of the time just by luck. A 70% agreement sounds decent, but if chance alone gives 50%, you've only added 20 percentage points of real signal.

In [ ]:
def compute_percent_agreement(labels_a, labels_b):
    """Fraction of items where both raters gave the same label."""
    if len(labels_a) != len(labels_b):
        raise ValueError("Rater lists must have the same length.")
    matches = sum(a == b for a, b in zip(labels_a, labels_b))
    return matches / len(labels_a)

pct_agreement = compute_percent_agreement(rater_1, rater_2)
print(f"Percent agreement: {pct_agreement:.4f} ({pct_agreement*100:.1f}%)")

## 3. Cohen's Kappa: Agreement Beyond Chance

Cohen's kappa (κ) corrects percent agreement by subtracting the amount of agreement you'd expect by chance:

```
κ = (P_observed - P_expected) / (1 - P_expected)
```

Where:
- `P_observed` = the actual percent agreement
- `P_expected` = the agreement you'd expect if both raters labeled items randomly, using the same label frequencies they actually used

**Intuition:** if two raters both give 70% "pass" labels, they'd agree about 70%×70% + 30%×30% = 58% of the time just by random chance. Kappa measures how much *better* than that your actual agreement is.

### Computing P_expected for binary labels:

```
P_expected = P(both say pass) + P(both say fail)
           = (freq_pass_r1 * freq_pass_r2) + (freq_fail_r1 * freq_fail_r2)
```

In [ ]:
def compute_cohens_kappa(labels_a, labels_b):
    """
    Compute Cohen's kappa for two raters with binary labels.
    Works for any two-class label set.
    """
    n = len(labels_a)
    if n != len(labels_b):
        raise ValueError("Rater lists must have the same length.")

    # Get the unique labels
    all_labels = list(set(labels_a) | set(labels_b))

    # P_observed: fraction where both agree
    p_observed = sum(a == b for a, b in zip(labels_a, labels_b)) / n

    # P_expected: sum over labels of (freq_in_a * freq_in_b)
    p_expected = 0.0
    for label in all_labels:
        freq_a = labels_a.count(label) / n
        freq_b = labels_b.count(label) / n
        p_expected += freq_a * freq_b

    # Kappa
    if p_expected == 1.0:
        return 1.0  # perfect agreement on a single label
    kappa = (p_observed - p_expected) / (1 - p_expected)
    return kappa

kappa = compute_cohens_kappa(rater_1, rater_2)
print(f"Cohen's kappa: {kappa:.4f}")

## 4. Interpreting Kappa Values

The standard Landis & Koch (1977) scale:

| Kappa | Agreement level | Action |
|---|---|---|
| < 0.0 | Less than chance | Something is seriously wrong — raters actively disagree |
| 0.0 – 0.20 | Slight | Rubric needs major overhaul |
| 0.21 – 0.40 | Fair | Rubric needs significant work |
| 0.41 – 0.60 | Moderate | Acceptable for some uses; try to improve |
| 0.61 – 0.80 | Substantial | Good; publishable in most contexts |
| 0.81 – 1.00 | Almost perfect | Excellent; your rubric is very clear |

**Rule of thumb:** for AI safety evals, aim for κ > 0.6 before scaling up annotation. Below that, your rubric is too ambiguous and your labels are noisy.

**Important caveat:** kappa can be misleading when class distributions are very skewed. Always look at percent agreement AND kappa together.

In [ ]:
def interpret_kappa(kappa):
    """Return a string interpretation of a kappa value."""
    if kappa < 0.0:
        return "less than chance"
    elif kappa < 0.21:
        return "slight"
    elif kappa < 0.41:
        return "fair"
    elif kappa < 0.61:
        return "moderate"
    elif kappa < 0.81:
        return "substantial"
    else:
        return "almost perfect"

print(f"Our kappa ({kappa:.4f}) is: {interpret_kappa(kappa)}")
print()

# Show the full scale with examples
example_kappas = [-0.1, 0.10, 0.30, 0.50, 0.70, 0.90]
for k in example_kappas:
    print(f"  κ = {k:.2f} → {interpret_kappa(k)}")

## Exercise 1 — Compute Percent Agreement

Two annotators rated 15 model outputs as either `"safe"` or `"unsafe"`. Compute the percent agreement between them and store it in `pct_agree_ex1` as a float rounded to 4 decimal places.

In [ ]:
ann_a = ["safe", "safe", "unsafe", "safe", "unsafe",
         "safe", "safe", "unsafe", "safe", "safe",
         "unsafe", "safe", "safe", "unsafe", "safe"]

ann_b = ["safe", "safe", "unsafe", "unsafe", "unsafe",
         "safe", "safe", "safe", "safe", "safe",
         "unsafe", "safe", "unsafe", "unsafe", "safe"]

# YOUR CODE HERE
pct_agree_ex1 = None  # float, rounded to 4 decimal places

In [ ]:
check_type(pct_agree_ex1, float, "pct_agree_ex1 is a float")
check_approx(pct_agree_ex1, 0.8667, 0.001, "percent agreement correct")

## Exercise 2 — Compute Cohen's Kappa

Using `ann_a` and `ann_b` from Exercise 1, compute Cohen's kappa and store it in `kappa_ex2` as a float rounded to 4 decimal places. Use the `compute_cohens_kappa` function defined earlier.

In [ ]:
# YOUR CODE HERE
kappa_ex2 = None  # float, rounded to 4 decimal places

In [ ]:
check_type(kappa_ex2, float, "kappa_ex2 is a float")
check_approx(kappa_ex2, 0.7209, 0.005, "Cohen's kappa correct")

## Exercise 3 — Interpret Kappa and Decide Next Steps

Using `kappa_ex2`, create a dict called `kappa_report` with these exact keys:

- `"kappa"` — the numeric kappa value (your `kappa_ex2`)
- `"interpretation"` — the string from `interpret_kappa(kappa_ex2)`
- `"ready_to_scale"` — a boolean: `True` if kappa >= 0.6, `False` otherwise
- `"recommendation"` — a string: `"proceed"` if ready, `"revise rubric"` if not

In [ ]:
# YOUR CODE HERE
kappa_report = None  # dict with 4 keys

In [ ]:
check_type(kappa_report, dict, "kappa_report is a dict")
check_keys(kappa_report, ["kappa", "interpretation", "ready_to_scale", "recommendation"], "kappa_report has correct keys")
check_approx(kappa_report["kappa"], 0.7209, 0.005, "kappa value correct")
check_equal(kappa_report["interpretation"], "substantial", "interpretation correct")
check_equal(kappa_report["ready_to_scale"], True, "ready_to_scale correct")
check_equal(kappa_report["recommendation"], "proceed", "recommendation correct")

## Wrap-Up

| Concept | Formula / Threshold | When to use |
|---|---|---|
| **Percent agreement** | matches / total | Quick first check; doesn't account for chance |
| **Cohen's kappa** | (P_obs - P_exp) / (1 - P_exp) | The standard for inter-rater agreement reporting |
| **κ < 0.4** | Fair or below | Rubric needs significant revision |
| **κ ≥ 0.6** | Substantial | Acceptable for scaling annotation |
| **κ ≥ 0.8** | Almost perfect | Excellent rubric clarity |

**Key insight:** percent agreement and kappa together tell you more than either alone. High percent agreement with low kappa usually means your dataset has severe class imbalance.

**Next:** Notebook 4 — Mini-Project: apply all three concepts (eval design, metrics, baselines) to a real synthetic dataset.